# Mask2Former Training — Lane Detection

**Architecture:** Mask2Former + Swin-Small backbone  
**Dataset:** Lane Detection (5 classes, 1610 images)  
**Strategy:** Gradual freezing (3 phases) + CosineAnnealingWarmRestarts  
**Tracking:** MLflow → Render.com

---
**Before running:** Make sure Runtime → Change runtime type → GPU (T4)

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch: {torch.__version__}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers accelerate
!pip install -q mlflow pymongo
!pip install -q albumentations pycocotools
!pip install -q roboflow dvc python-dotenv
print('Dependencies installed')

In [ ]:
# ── Cell 3: Load secrets from Colab ─────────────────────────────────────────
# Add these in Colab: left sidebar → 🔑 Secrets
#   ROBOFLOW_API_KEY, HF_TOKEN, MONGO_URI, MLFLOW_TRACKING_URI
import os
from google.colab import userdata

os.environ['ROBOFLOW_API_KEY']    = userdata.get('ROBOFLOW_API_KEY')
os.environ['ROBOFLOW_WORKSPACE']  = 'test-mfeql'
os.environ['ROBOFLOW_PROJECT']    = 'lane-detection-segmentation-edyqp-fibkz'
os.environ['ROBOFLOW_VERSION']    = '1'
os.environ['HF_TOKEN']            = userdata.get('HF_TOKEN')
os.environ['HF_REPO_ID']          = 'srnortw/mask2former-lane-seg'
os.environ['MONGO_URI']           = userdata.get('MONGO_URI')
os.environ['MONGO_DB_NAME']       = 'mask2former'
os.environ['MONGO_COLLECTION_PREDICTIONS'] = 'predictions'
os.environ['MONGO_COLLECTION_DRIFT']       = 'drift_reports'
os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME']  = 'srnortw'
os.environ['MLFLOW_TRACKING_PASSWORD']  = userdata.get('MLFLOW_TRACKING_PASSWORD')
os.environ['MLFLOW_EXPERIMENT_NAME']    = 'mask2former-swin'
os.environ['GITHUB_TOKEN']        = ''
os.environ['GITHUB_REPO']         = 'srnortw/mask2former'
print('Secrets loaded')

In [ ]:
# ── Cell 4: Clone repo and download data ────────────────────────────────────
import os

if not os.path.exists('mask2former'):
    !git clone https://github.com/srnortw/mask2former.git

%cd mask2former

# Download dataset from Roboflow (faster than dvc pull for Colab)
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace('test-mfeql').project('lane-detection-segmentation-edyqp-fibkz')
dataset = project.version(1).download('coco-segmentation', location='data/raw', overwrite=False)
print(f'Data ready at: {dataset.location}')

In [ ]:
# ── Cell 5: Load config and verify data ──────────────────────────────────────
import sys, os
sys.path.insert(0, '/content/mask2former/src')
os.chdir('/content/mask2former')

from config_loader import load_config
from data.dataset import build_dataloaders

# Colab: reduce workers to 2
cfg = load_config()

train_loader, val_loader = build_dataloaders(cfg)
print(f'Train: {len(train_loader.dataset)} samples')
print(f'Val:   {len(val_loader.dataset)} samples')

# Quick batch check
import torch
imgs, targets = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}')
print(f'Sample masks: {targets[0]["masks"].shape}')

In [ ]:
# ── Cell 6: Build model + verify freezing ────────────────────────────────────
import sys
sys.path.insert(0, '/content/mask2former/src')
from models.mask2former import build_model, set_phase

device = torch.device('cuda')
model = build_model(cfg).to(device)

# Show phase 1 trainable params
set_phase(model, phase=1)

# Sanity forward pass
model.eval()
with torch.no_grad():
    out = model(pixel_values=imgs.to(device))
print(f'Output keys: {list(out.keys())}')
print(f'Masks shape: {out.masks_queries_logits.shape}')
print(f'Logits shape: {out.class_queries_logits.shape}')

In [ ]:
# ── Cell 7: Check MLflow connection ─────────────────────────────────────────
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'])

# Test connection with a quick log
with mlflow.start_run(run_name='connection-test') as run:
    mlflow.log_param('test', True)
    mlflow.log_metric('test_metric', 1.0)
    print(f'MLflow run ID: {run.info.run_id}')
    print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print('MLflow connection OK')

In [ ]:
# ── Cell 8: TRAIN ───────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/mask2former/src')
from train import train

best_map = train(cfg)
print(f'\nFinal best mAP: {best_map:.4f}')

In [ ]:
# ── Cell 9: Push checkpoint to Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/mask2former-mlops/checkpoints'
os.makedirs(dst, exist_ok=True)

for ckpt in ['best_model.pth', 'phase1_final.pth', 'phase2_final.pth', 'phase3_final.pth']:
    src = f'checkpoints/{ckpt}'
    if os.path.exists(src):
        shutil.copy(src, f'{dst}/{ckpt}')
        print(f'Copied: {ckpt}')

print('Checkpoints backed up to Drive')

In [ ]:
# ── Cell 10: Push checkpoint to Hugging Face Hub ─────────────────────────────
from huggingface_hub import HfApi, create_repo

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_REPO_ID']

create_repo(repo_id, repo_type='model', private=True, exist_ok=True)

api.upload_file(
    path_or_fileobj='checkpoints/best_model.pth',
    path_in_repo='best_model.pth',
    repo_id=repo_id,
)
print(f'Model pushed to: https://huggingface.co/{repo_id}')